In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier

# load data
df = pd.read_csv('edited.csv')
print(df['phrase'].nunique())

# drop duplicate phrases to prevent data leakage
df = df.drop_duplicates(subset=['phrase']).reset_index(drop=True)

X = df["phrase"]
y = df["prompt"]

# train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# define models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Linear SVM": LinearSVC(dual="auto", random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Multinomial Naive Bayes": MultinomialNB(),
    "K-Nearest Neighbors": KNeighborsClassifier()
}

# train and eval
for name, model in models.items():
  pipeline = Pipeline([
      ('tfidf', TfidfVectorizer(ngram_range=(1,2), stop_words='english')),
      ('clf', model)
  ])
  pipeline.fit(X_train, y_train)
  print(f"Model: {name}")
  y_pred = pipeline.predict(X_test)
  print(classification_report(y_test, y_pred))



706
Model: Logistic Regression
                    precision    recall  f1-score   support

              Acne       0.88      1.00      0.93         7
         Back pain       1.00      0.40      0.57         5
     Blurry vision       0.67      0.40      0.50         5
   Body feels weak       0.50      0.60      0.55         5
             Cough       0.62      0.83      0.71         6
          Ear ache       1.00      1.00      1.00         6
    Emotional pain       1.00      0.40      0.57         5
      Feeling cold       0.83      0.83      0.83         6
     Feeling dizzy       1.00      0.80      0.89         5
         Foot ache       1.00      0.80      0.89         5
  Hair falling out       0.83      1.00      0.91         5
    Hard to breath       0.33      0.20      0.25         5
         Head ache       0.62      0.83      0.71         6
       Heart hurts       0.80      0.67      0.73         6
    Infected wound       0.31      0.57      0.40         7
Injury f

In [2]:
#CNN
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import LabelEncoder

tf.random.set_seed(42)

# 1. labels -> integers (Keras needs 0..24, not category names)
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)
num_classes = len(le.classes_)

# 2. text -> padded sequences of word IDs (vocabulary learned from train only)
max_len = 30
vectorizer = layers.TextVectorization(max_tokens=10000, output_sequence_length=max_len)
vectorizer.adapt(tf.constant(X_train.tolist()))
X_train_seq = vectorizer(tf.constant(X_train.tolist())).numpy()
X_test_seq = vectorizer(tf.constant(X_test.tolist())).numpy()

# 3. build the CNN
cnn = keras.Sequential([
    keras.Input(shape=(max_len,)),
    layers.Embedding(vectorizer.vocabulary_size(), 64),       # learned word vectors
    layers.Conv1D(64, 3, activation="relu", padding="same"),  # detects 3-word patterns
    layers.GlobalMaxPooling1D(),                              # keeps strongest signal per filter
    layers.Dropout(0.4),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(num_classes, activation="softmax"),
])
cnn.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

# 4. train (holds out 15% of train for early stopping)
early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
cnn.fit(X_train_seq, y_train_enc, validation_split=0.15,
        epochs=50, batch_size=32, callbacks=[early_stop])

# 5. evaluate the same way as your other models
y_pred = le.inverse_transform(cnn.predict(X_test_seq).argmax(axis=1))
print("Model: 1D CNN")
print(classification_report(y_test, y_pred))

Epoch 1/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - accuracy: 0.0459 - loss: 3.2193 - val_accuracy: 0.0235 - val_loss: 3.2245
Epoch 2/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.0647 - loss: 3.2012 - val_accuracy: 0.0471 - val_loss: 3.2227
Epoch 3/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.0731 - loss: 3.1911 - val_accuracy: 0.0353 - val_loss: 3.2203
Epoch 4/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1253 - loss: 3.1705 - val_accuracy: 0.0706 - val_loss: 3.2134
Epoch 5/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1211 - loss: 3.1550 - val_accuracy: 0.1059 - val_loss: 3.2037
Epoch 6/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1420 - loss: 3.1137 - val_accuracy: 0.1294 - val_loss: 3.1867
Epoch 7/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1837 - loss: 3.0562 - val_accuracy: 0.1529 - val_loss: 3.1630
Epoch 8/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.2234 - loss: 2.9754 - val_accuracy: 0.1647 - val_loss

In [3]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier

# load data
df = pd.read_csv('edited.csv')
print(df['phrase'].nunique())

# without using duplicates
# df = df.drop_duplicates(subset=['phrase']).reset_index(drop=True)

X = df["phrase"]
y = df["prompt"]

# train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# define models
models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Linear SVM": LinearSVC(dual="auto", random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Multinomial Naive Bayes": MultinomialNB(),
    "K-Nearest Neighbors": KNeighborsClassifier()
}

# train and eval
# ngram_range groups phrases
for name, model in models.items():
  pipeline = Pipeline([
      ('tfidf', TfidfVectorizer(ngram_range=(1,2), stop_words='english')),
      ('clf', model)
  ])
  pipeline.fit(X_train, y_train)
  print(f"Model: {name}")
  y_pred = pipeline.predict(X_test)
  print(classification_report(y_test, y_pred))



706
Model: Logistic Regression
                    precision    recall  f1-score   support

              Acne       1.00      1.00      1.00        66
         Back pain       1.00      1.00      1.00        52
     Blurry vision       1.00      0.96      0.98        49
   Body feels weak       1.00      0.98      0.99        48
             Cough       1.00      1.00      1.00        59
          Ear ache       1.00      1.00      1.00        54
    Emotional pain       1.00      1.00      1.00        46
      Feeling cold       1.00      1.00      1.00        53
     Feeling dizzy       1.00      1.00      1.00        57
         Foot ache       1.00      1.00      1.00        44
  Hair falling out       0.98      1.00      0.99        53
    Hard to breath       1.00      1.00      1.00        47
         Head ache       0.95      1.00      0.97        53
       Heart hurts       1.00      1.00      1.00        55
    Infected wound       0.97      1.00      0.98        61
Injury f